In [1]:
# Goal: Use the DeFiLlama API to pull data on multiple tokens and use the TVL available via defillama (for free and without a key!) to create some exceptional interactive data visualizations with plotly.express (px)

In [1]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

import plotly.express as px

import requests
import urllib3
import json
import datetime as dt
import urllib
import urllib.parse
import time

from datetime import datetime, timedelta
from requests import Request, Session
from requests.exceptions import ConnectionError, Timeout, TooManyRedirects
from requests.packages.urllib3.exceptions import InsecureRequestWarning

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

import warnings
warnings.simplefilter('ignore')

In [2]:
# DFL Library import & API init:
from defillama import DefiLlama

llama = DefiLlama()

In [6]:
# help menu on 'llama' will show all of the built-in methods available via the dfl wrapper:
help(llama)

Help on DefiLlama in module defillama.defillama object:

class DefiLlama(builtins.object)
 |  DeFi Llama class to act as DeFi Llama's API client.
 |  All the requests can be made through this class.
 |  
 |  Methods defined here:
 |  
 |  __init__(self)
 |      Initialize the object
 |  
 |  get_all_protocols(self)
 |      Returns basic information on all listed protocols, their current
 |      TVL and the changes to it in the last hour/day/week.
 |      Endpoint: GET /protocols
 |      
 |      :return: JSON response
 |  
 |  get_batch_historical_prices(self, coins: str, searchWidth: str = '600')
 |      Get historical prices of tokens by contract address
 |  
 |  get_chain_dexs(self, chain, excludeTotalDataChart=True, excludeTotalDataChartBreakdown=True, dataType='dailyVolume')
 |      list all dexs filter by chain
 |  
 |  get_chain_options_dexs(self, chain, excludeTotalDataChart=True, excludeTotalDataChartBreakdown=True, dataType='dailyVolume')
 |      list all options dexs
 |  
 |

In [3]:
# Pull all historical data for Solana (entire chain); this will be used in our viz as a benchmark;

solana_data = llama.get_historical_tvl_chain('Solana')
solana_data

[{'date': 1616025600, 'tvl': 148988798},
 {'date': 1616112000, 'tvl': 153204289},
 {'date': 1616198400, 'tvl': 147690914},
 {'date': 1616284800, 'tvl': 151935325},
 {'date': 1616371200, 'tvl': 152981122},
 {'date': 1616457600, 'tvl': 162065403},
 {'date': 1616544000, 'tvl': 156101311},
 {'date': 1616630400, 'tvl': 144439504},
 {'date': 1616716800, 'tvl': 135610450},
 {'date': 1616803200, 'tvl': 147786656},
 {'date': 1616889600, 'tvl': 172583244},
 {'date': 1616976000, 'tvl': 189014228},
 {'date': 1617062400, 'tvl': 202157669},
 {'date': 1617148800, 'tvl': 209794033},
 {'date': 1617235200, 'tvl': 216074736},
 {'date': 1617321600, 'tvl': 236083655},
 {'date': 1617408000, 'tvl': 236307232},
 {'date': 1617494400, 'tvl': 236512420},
 {'date': 1617580800, 'tvl': 236284645},
 {'date': 1617667200, 'tvl': 272660066},
 {'date': 1617753600, 'tvl': 268791821},
 {'date': 1617840000, 'tvl': 279546575},
 {'date': 1617926400, 'tvl': 294784827},
 {'date': 1618012800, 'tvl': 331055199},
 {'date': 161809

In [4]:
# Since the data has no extra 'fluff' in the top of the raw json, can just load directly into a dataframe:
df_sol = pd.DataFrame(solana_data)

# The timestamps are in UNIX, so need to convert to datetime in a human-readable format, then set as the index and sort by the date:

df_sol['date'] = pd.to_datetime(df_sol['date'], unit='s')
df_sol.set_index('date', inplace=True)
df_sol.sort_index(inplace=True)

# Take a look at the df from a few angles:
print(df_sol.shape)
print(df_sol.info())

df_sol.head(3)

(1779, 1)
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1779 entries, 2021-03-18 to 2026-01-29
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   tvl     1779 non-null   int64
dtypes: int64(1)
memory usage: 27.8 KB
None


,tvl
date,
2021-03-18,148988798
2021-03-19,153204289
2021-03-20,147690914


In [11]:
# Can see all of the protocols available:
llama.get_all_protocols()

[{'id': '2269',
  'name': 'Binance CEX',
  'address': None,
  'symbol': '-',
  'url': 'https://www.binance.com',
  'description': 'Binance is a cryptocurrency exchange which is the largest exchange in the world in terms of daily trading volume of cryptocurrencies',
  'chain': 'Multi-Chain',
  'logo': 'https://icons.llama.fi/binance-cex.jpg',
  'audits': '0',
  'gecko_id': 'binancecoin',
  'cmcId': None,
  'category': 'CEX',
  'chains': ['Ethereum',
   'Bitcoin',
   'Binance',
   'Ripple',
   'Tron',
   'Solana',
   'Doge',
   'Arbitrum',
   'Base',
   'Avalanche',
   'Litecoin',
   'TON',
   'Hedera',
   'Near',
   'Polygon',
   'Aptos',
   'Optimism',
   'Chiliz',
   'Sonic',
   'Plasma',
   'Op_Bnb',
   'Stellar',
   'Algorand',
   'Celo',
   'Ronin',
   'zkSync Era',
   'Manta',
   'Starknet',
   'Scroll',
   'Polkadot',
   'Sui',
   'Fantom'],
  'module': 'binance/index.js',
  'twitter': 'binance',
  'listedAt': 1668170565,
  'methodology': 'We collect the wallets from this Binance

In [5]:
# Just grabbing a semi-random protocol (selected Jito as it was the first well known one I could find; again, this is really for the viz, so any example with enough data should be GTG!)

response = llama.get_protocol('Jito Liquid Staking')
response

{'id': '2308',
 'name': 'Jito Liquid Staking',
 'address': 'solana:jtojtomepa8beP8AuQc6eXt5FriJwfFMwQx2v2f9mCL',
 'symbol': 'JTO',
 'url': 'https://jito.network',
 'description': 'JitoSOL is the first liquid staking derivative on Solana to include MEV rewards. Tracks the price of SOL while accruing staking and MEV rewards. Yield is accrued in the price so it will steadily appreciate vs. SOL',
 'chain': 'Solana',
 'logo': 'https://icons.llama.fi/jito-liquid-staking.png',
 'audits': '2',
 'gecko_id': None,
 'cmcId': None,
 'category': 'Liquid Staking',
 'chains': ['Solana'],
 'module': 'jito/index.js',
 'twitter': 'jito_sol',
 'audit_links': ['https://spl.solana.com/stake-pool#security-audits',
  'https://2926710696-files.gitbook.io/~/files/v0/b/gitbook-x-prod.appspot.com/o/spaces%2Ffrb9MGTK6eZJlEQJyylq%2Fuploads%2F1jfEDpGcd5YlnHusbKYO%2FNeodymeJito.pdf'],
 'oraclesBreakdown': [{'name': 'Switchboard',
   'type': 'Secondary',
   'proof': ['https://github.com/DefiLlama/defillama-server/pul

In [6]:
# Extract the TVL data 

# *The raw tvl data has a bunch of headers and fluff so we have to start lower down;)

tvl_data = response['chainTvls']['Solana']['tvl']

# Create an empty list to store the processed data:

processed_data = []

for entry in tvl_data:
    row = {
        'date': datetime.fromtimestamp(entry['date']).strftime('%Y-%m-%d')
        , 'totalLiquidityUSD': entry['totalLiquidityUSD']
    }
                                        
    # Handle token breakdowns if present:
    if 'tokens' in entry:
        for token, amount in entry['tokens'].items():
            row[f'token_{token}'] = amount
            
    processed_data.append(row)
    
# Create the dataframe (for the token whose TVL/ data will be overlaid on the Solana baseline):
df_token = pd.DataFrame(processed_data)

# Set 'date' as the index & sort (this is optional, but DateTimeIndex is helpful for time-series analysis):
df_token.set_index('date', inplace=True)
df_token = df_token.sort_index()

# The usual dataframe sneak-a-peak operations to see shape, info/dtypes/nulls & first few rows:
print(df_token.shape)
print(df_token.info())

df_token.head(3)

(1163, 1)
<class 'pandas.core.frame.DataFrame'>
Index: 1163 entries, 2022-11-23 to 2026-01-28
Data columns (total 1 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   totalLiquidityUSD  1163 non-null   int64
dtypes: int64(1)
memory usage: 18.2+ KB
None


,totalLiquidityUSD
date,
2022-11-23,4515393
2022-11-24,4467877
2022-11-25,4472156


In [9]:
# Create a column to show day-over-day percent change:

def add_percent_change(df, tvl_column):
    
    # Sort the dataframe by date:
    df = df.sort_index()
    
    # Calculate % change:
    df['TVL_percent_change'] = df[tvl_column].pct_change() * 100
    
    # Format the % change to two decimal places (avoids unnecessary clutter):
    df['TVL_percent_change'] = df['TVL_percent_change']\
            .apply(lambda x: f"{x:.2f}" if pd.notnull(x) else None)
    
    return df

In [10]:
df_token = add_percent_change(df_token, 'totalLiquidityUSD')
df_token.head(3)

,totalLiquidityUSD,TVL_percent_change
date,,
2022-11-23,4515393,None
2022-11-24,4467877,-1.05
2022-11-25,4472156,0.10


In [11]:
df_token.columns

Index(['totalLiquidityUSD', 'TVL_percent_change'], dtype='object')

In [12]:
# Built a line plot w/ plotly express for the token

fig_token = px.line(
    df_token
    , x=df_token.index
    , y='totalLiquidityUSD'
    , width=1200, height=900
    , title='Total Value Locked (TVL) in USD, over time for Jito Liquid Staking' 
    , labels={
        'totalLiquidityUSD': 'Total Liquidity (USD)'
        , 'date': 'Date'
        , 'TVL_percent_change': '% Change'
    }
    , template='plotly'
    , markers=True
    , color_discrete_sequence=['red']
    , hover_data={'TVL_percent_change': True}
)

# Improve the layout
fig_token.update_layout(
    xaxis_title='Date'
    , yaxis_title='Total Value Locked [TVL] (USD)'
    , hovermode='x unified'
    , yaxis_tickformat='$,.0f'
)

fig_token.update_xaxes(
    rangeslider_visible=True
)

fig_token.update_traces(
    line=dict(
        width=3)
)

fig_token.show()